# DementiaSense: ML-Based Early Dementia Risk Screening and Decision Support System
## Stages 2-6: Data Acquisition, Preparation, Preprocessing, Balancing and EDA

**How to use this notebook**
1. Download the dataset from Kaggle (`rabieelkharoua/alzheimers-disease-dataset`) and put the CSV next to this notebook. Edit `DATA_PATH` in the first code cell if your file name differs.
2. Run top to bottom. Every stage ends with a **What / Why / How / Result** cell. Those cells are **yours to write** after you have seen the real outputs. Do not copy conclusions from anywhere else, because the guidelines treat unexplained or fabricated results as misconduct.
3. The **test set is created in Stage 4 and not touched again** until final evaluation (Stage 11). Everything after the split (leakage check, balancing experiment, EDA) uses the **training set only**.

In [ ]:
import warnings, random, platform, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

try:
    from IPython.display import display
except ImportError:          # running as a plain script
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------- Reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------- Paths (edit if needed) ----------
DATA_PATH = Path("alzheimers_disease_data.csv")
OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| seed", SEED)

---
# Stage 2: Data Acquisition & Understanding

**Mandatory items:** source, ownership/licence, size, features, target, data types, limitations.

| Item | Fill in from the Kaggle page (do not guess) |
|---|---|
| Dataset name and URL | Alzheimer's Disease Dataset, `https://www.kaggle.com/datasets/rabieelkharoua/alzheimers-disease-dataset` |
| Author / owner | |
| Licence | (one mirror lists CC BY 4.0; confirm on the Kaggle page) |
| How the data was created | **Read the description. State clearly whether it is real clinical data or synthetic.** |
| Date accessed | |

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Shape (rows, columns):", raw.shape)
display(raw.head())
print()
raw.info()

In [ ]:
# ---------- Data dictionary (single source of truth for column groups) ----------
# Descriptions follow the dataset's documentation as I recall it. VERIFY each row against the Kaggle page.
dd = pd.DataFrame([
    # feature, group, kind, description
    ("PatientID", "id", "id", "Unique patient identifier"),
    ("Age", "demographic", "continuous", "Age in years"),
    ("Gender", "demographic", "binary", "0 = Male, 1 = Female"),
    ("Ethnicity", "demographic", "nominal", "0 Caucasian, 1 African American, 2 Asian, 3 Other"),
    ("EducationLevel", "demographic", "ordinal", "0 None, 1 High School, 2 Bachelor's, 3 Higher"),
    ("BMI", "lifestyle", "continuous", "Body mass index"),
    ("Smoking", "lifestyle", "binary", "0 = No, 1 = Yes"),
    ("AlcoholConsumption", "lifestyle", "continuous", "Weekly alcohol units"),
    ("PhysicalActivity", "lifestyle", "continuous", "Weekly physical activity (hours)"),
    ("DietQuality", "lifestyle", "continuous", "Diet quality score"),
    ("SleepQuality", "lifestyle", "continuous", "Sleep quality score"),
    ("FamilyHistoryAlzheimers", "history", "binary", "Family history of Alzheimer's"),
    ("CardiovascularDisease", "history", "binary", "History of cardiovascular disease"),
    ("Diabetes", "history", "binary", "History of diabetes"),
    ("Depression", "history", "binary", "History of depression"),
    ("HeadInjury", "history", "binary", "History of head injury"),
    ("Hypertension", "history", "binary", "History of hypertension"),
    ("SystolicBP", "clinical", "continuous", "Systolic blood pressure (mmHg)"),
    ("DiastolicBP", "clinical", "continuous", "Diastolic blood pressure (mmHg)"),
    ("CholesterolTotal", "clinical", "continuous", "Total cholesterol (mg/dL)"),
    ("CholesterolLDL", "clinical", "continuous", "LDL cholesterol (mg/dL)"),
    ("CholesterolHDL", "clinical", "continuous", "HDL cholesterol (mg/dL)"),
    ("CholesterolTriglycerides", "clinical", "continuous", "Triglycerides (mg/dL)"),
    ("MMSE", "cognitive_test", "continuous", "Mini-Mental State Examination, 0-30 (lower = worse)"),
    ("FunctionalAssessment", "cognitive_test", "continuous", "Functional assessment score (lower = worse)"),
    ("ADL", "cognitive_test", "continuous", "Activities of Daily Living score (lower = more impaired)"),
    ("MemoryComplaints", "symptom", "binary", "Memory complaints present"),
    ("BehavioralProblems", "symptom", "binary", "Behavioural problems present"),
    ("Confusion", "symptom", "binary", "Confusion present"),
    ("Disorientation", "symptom", "binary", "Disorientation present"),
    ("PersonalityChanges", "symptom", "binary", "Personality changes present"),
    ("DifficultyCompletingTasks", "symptom", "binary", "Difficulty completing tasks"),
    ("Forgetfulness", "symptom", "binary", "Forgetfulness present"),
    ("Diagnosis", "target", "binary", "0 = No Alzheimer's diagnosis, 1 = Diagnosis"),
    ("DoctorInCharge", "admin", "id", "Administrative field (not clinically meaningful)"),
], columns=["feature", "group", "kind", "description"])

# ---------- Check that my dictionary matches the real file ----------
not_in_file = sorted(set(dd.feature) - set(raw.columns))
not_in_dict = sorted(set(raw.columns) - set(dd.feature))
if not_in_file or not_in_dict:
    raise ValueError(f"Column mismatch. In dictionary but not in file: {not_in_file}. "
                     f"In file but not in dictionary: {not_in_dict}. Edit the dictionary above.")
print("Column names match the data dictionary.")

TARGET = "Diagnosis"
CONTINUOUS = dd.query("kind == 'continuous'").feature.tolist()
BINARY     = dd.query("kind == 'binary' and group != 'target'").feature.tolist()
ORDINAL    = dd.query("kind == 'ordinal'").feature.tolist()
NOMINAL    = dd.query("kind == 'nominal'").feature.tolist()
ALL_FEATURES = CONTINUOUS + BINARY + ORDINAL + NOMINAL

grp = lambda *g: dd[dd.group.isin(g)].feature.tolist()
FEATURE_SETS = {
    # A: what a GP/community worker could collect without cognitive testing
    "A_screening": grp("demographic", "lifestyle", "history", "clinical"),
    # B: adds reported symptoms
    "B_plus_symptoms": grp("demographic", "lifestyle", "history", "clinical", "symptom"),
    # C: adds cognitive/functional test scores (decision-support setting; closest to the diagnosis itself)
    "C_plus_cognitive_tests": grp("demographic", "lifestyle", "history", "clinical", "symptom", "cognitive_test"),
}
assert set(FEATURE_SETS["C_plus_cognitive_tests"]) == set(ALL_FEATURES)

print(f"{len(ALL_FEATURES)} candidate features | continuous {len(CONTINUOUS)}, binary {len(BINARY)}, "
      f"ordinal {len(ORDINAL)}, nominal {len(NOMINAL)}")
for k, v in FEATURE_SETS.items():
    print(f"  {k}: {len(v)} features")
display(dd)

In [ ]:
# ---------- Target variable ----------
tgt = raw[TARGET].value_counts(dropna=False).rename("count").to_frame()
tgt["percent"] = (tgt["count"] / len(raw) * 100).round(2)
display(tgt)

### Stage 2: What / Why / How / Result  *(write this yourself)*
- **What:** (dataset, size, features, target)
- **Why this dataset:** (size, public availability, feature types suit a *screening* tool, licence)
- **Limitations to discuss:** real vs synthetic origin; single row per patient so no longitudinal information; age range limited to older adults; population/geography unknown; features such as MMSE and ADL are assessment scores that sit close to the diagnosis (leakage risk, tested in Stage 4); `DoctorInCharge` carries no clinical meaning.
- **Result:** (rows x columns, target definition, feature groups)

---
# Stage 3: Data Preparation

Goal: a clean, correctly typed dataset with a documented record of every change in size.
Checks: duplicates (rows and IDs), identifier handling, data types, impossible values.

In [ ]:
shape_log = []
def log_shape(step, d):
    shape_log.append({"step": step, "rows": d.shape[0], "columns": d.shape[1]})

df = raw.copy()
log_shape("1. Raw file loaded", df)

# ---------- Duplicate analysis ----------
n_dup_rows = df.duplicated().sum()
n_dup_ids = df["PatientID"].duplicated().sum()
n_dup_ignoring_id = df.drop(columns=["PatientID"]).duplicated().sum()
print(f"Exact duplicate rows:                 {n_dup_rows}")
print(f"Duplicate PatientID values:           {n_dup_ids}")
print(f"Duplicate rows ignoring PatientID:    {n_dup_ignoring_id}")

# One row per patient? This decides whether a plain stratified split is safe (no patient in both train and test).
ONE_ROW_PER_PATIENT = (n_dup_ids == 0)
print("One row per patient:", ONE_ROW_PER_PATIENT)

df = df.drop_duplicates()
log_shape("2. After removing exact duplicates", df)

In [ ]:
# ---------- Identifiers / administrative columns ----------
print("DoctorInCharge unique values:", df["DoctorInCharge"].nunique())
df = df.drop(columns=["PatientID", "DoctorInCharge"])
log_shape("3. After dropping ID and administrative columns", df)

# ---------- Data types ----------
before_dtypes = df.dtypes.copy()
for c in ALL_FEATURES + [TARGET]:
    converted = pd.to_numeric(df[c], errors="coerce")
    newly_missing = converted.isna().sum() - df[c].isna().sum()
    if newly_missing > 0:
        print(f"WARNING: {c}: {newly_missing} values could not be parsed as numbers")
    df[c] = converted
print("Non-numeric columns remaining:", df.select_dtypes(exclude="number").columns.tolist())

# ---------- Domain/validity checks ----------
# Only definitional or very generous bounds. Violations are REPORTED, not silently fixed.
bounds = {"Age": (18, 110), "BMI": (10, 70), "SystolicBP": (70, 250), "DiastolicBP": (40, 150),
          "CholesterolTotal": (80, 500), "CholesterolLDL": (20, 400), "CholesterolHDL": (10, 150),
          "CholesterolTriglycerides": (20, 1000), "MMSE": (0, 30)}
rows = []
for c, (lo, hi) in bounds.items():
    rows.append({"feature": c, "allowed": f"{lo}-{hi}", "observed_min": df[c].min(),
                 "observed_max": df[c].max(), "violations": int(((df[c] < lo) | (df[c] > hi)).sum())})
display(pd.DataFrame(rows))

bad_binary = {c: sorted(set(df[c].dropna().unique()) - {0, 1}) for c in BINARY + [TARGET]}
bad_binary = {k: v for k, v in bad_binary.items() if v}
print("Binary columns containing values other than 0/1:", bad_binary or "none")

In [ ]:
log_shape("4. Prepared dataset (end of Stage 3)", df)
display(pd.DataFrame(shape_log))
display(df.describe().T.round(2))

### Stage 3: What / Why / How / Result  *(write this yourself)*
- **What / How:** (duplicate checks, IDs dropped, types converted, validity checks)
- **Why:** identifiers carry no clinical signal and can let a model memorise rows; duplicates inflate scores if copies land in both train and test.
- **Result:** (rows/columns before and after, number of duplicates found, any invalid values found)

---
# Stage 4: Data Preprocessing

Order of work (this order prevents leakage):
1. Inspect missing values, outliers and inconsistencies (read-only analysis).
2. **Split** into train and test (stratified, fixed seed).
3. Build the preprocessing **pipeline** and fit it **on the training data only**.
4. Leakage check on the training set.

In [ ]:
# ---------- 4.1 Missing-value analysis ----------
miss = pd.DataFrame({"missing": df.isna().sum(), "percent": (df.isna().mean() * 100).round(2)})
miss = miss[miss.missing > 0]
print("Columns with missing values:" if len(miss) else "No missing values found.")
if len(miss):
    display(miss)

# Disguised missing values (sentinels such as -1, 99, 999) in continuous columns
sentinel_hits = {c: int(df[c].isin([-1, 99, 999, 9999]).sum()) for c in CONTINUOUS}
print("Possible sentinel codes (-1/99/999/9999) per continuous column:",
      {k: v for k, v in sentinel_hits.items() if v} or "none")

In [ ]:
# ---------- 4.2 Outlier analysis (IQR rule) ----------
def iqr_summary(d, cols, k=1.5):
    out = []
    for c in cols:
        q1, q3 = d[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - k * iqr, q3 + k * iqr
        n_out = int(((d[c] < lo) | (d[c] > hi)).sum())
        out.append({"feature": c, "lower_fence": round(lo, 2), "upper_fence": round(hi, 2),
                    "n_outliers": n_out, "percent": round(n_out / len(d) * 100, 2)})
    return pd.DataFrame(out).sort_values("percent", ascending=False)

display(iqr_summary(df, CONTINUOUS))

n = len(CONTINUOUS); ncols = 5; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 2.6 * nrows))
for ax, c in zip(axes.ravel(), CONTINUOUS):
    sns.boxplot(x=df[c], ax=ax, color="#8ab4d9", fliersize=2)
    ax.set_xlabel(c)
for ax in axes.ravel()[n:]:
    ax.axis("off")
savefig("04_outlier_boxplots")

**Outlier decision rule (state your own decision after reading the table above):** clinical scores and lab values are bounded by their measurement scale, so a value flagged by the IQR rule is not automatically an error. Remove or cap a value only if it is **impossible** (Stage 3 validity checks). If outliers are few and plausible, keeping them is the defensible choice; scaling and tree-based models are tolerant of them. Write down what you found and what you decided.

In [ ]:
# ---------- 4.3 Train/test split (before any fitting) ----------
X = df[ALL_FEATURES].copy()
y = df[TARGET].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

split_tbl = pd.DataFrame({
    "rows": [len(y_train), len(y_test)],
    "positives (Diagnosis=1)": [int(y_train.sum()), int(y_test.sum())],
    "positive rate %": [round(y_train.mean() * 100, 2), round(y_test.mean() * 100, 2)],
}, index=["train", "test (held out until Stage 11)"])
display(split_tbl)
print("One row per patient, so a plain stratified split cannot put one patient in both sets:", ONE_ROW_PER_PATIENT)
print("Validation strategy for later stages: 5-fold StratifiedKFold inside the training set.")

X_train.assign(Diagnosis=y_train).to_csv(OUT_DIR / "train.csv", index=False)
X_test.assign(Diagnosis=y_test).to_csv(OUT_DIR / "test.csv", index=False)

In [ ]:
# ---------- 4.4 Preprocessing pipeline (fitted on TRAIN only) ----------
def build_preprocessor(features):
    cont = [c for c in CONTINUOUS if c in features]
    binr = [c for c in BINARY if c in features]
    ordn = [c for c in ORDINAL if c in features]
    nomn = [c for c in NOMINAL if c in features]
    parts = []
    if cont:   # median imputation is robust to skew/outliers; scaling helps LR/SVM/KNN/NN, harmless for trees
        parts.append(("cont", Pipeline([("imp", SimpleImputer(strategy="median")),
                                        ("sc", StandardScaler())]), cont))
    if binr:   # already 0/1: only impute (most frequent)
        parts.append(("bin", SimpleImputer(strategy="most_frequent"), binr))
    if ordn:   # ordered codes: keep order, scale
        parts.append(("ord", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                       ("sc", StandardScaler())]), ordn))
    if nomn:   # unordered category codes: one-hot (no false ordering)
        parts.append(("nom", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                       ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), nomn))
    return ColumnTransformer(parts, remainder="drop", verbose_feature_names_out=False)

prep = build_preprocessor(ALL_FEATURES)
Xt_train = prep.fit_transform(X_train)          # fit on train only
Xt_test = prep.transform(X_test)                # test is only transformed
names_after = prep.get_feature_names_out()

print(f"Features before preprocessing: {X_train.shape[1]}")
print(f"Features after preprocessing:  {Xt_train.shape[1]}  (Ethnicity expanded by one-hot encoding)")
print(f"Train matrix {Xt_train.shape} | Test matrix {Xt_test.shape}")
print("Any NaN left in train matrix:", bool(np.isnan(Xt_train).any()))

log_shape("5. Train design matrix after preprocessing", pd.DataFrame(Xt_train))
display(pd.DataFrame(shape_log))

In [ ]:
# ---------- 4.5 Data-leakage check (TRAIN set only) ----------
# (a) Single-feature predictive power. A single column that separates the classes almost perfectly is
#     either a leak or a near-definition of the label (e.g. a score used to make the diagnosis).
rows = []
for c in ALL_FEATURES:
    xi = X_train[c].fillna(X_train[c].median())
    auc = roc_auc_score(y_train, xi)
    group = dd.loc[dd.feature == c, "group"].iloc[0]
    rows.append({"feature": c, "group": group, "auc": round(auc, 3), "strength": round(max(auc, 1 - auc), 3)})
leak = pd.DataFrame(rows).sort_values("strength", ascending=False).reset_index(drop=True)
display(leak.head(12))

SUSPICIOUS = 0.75
print("Features with single-feature AUC strength >=", SUSPICIOUS, ":",
      leak.loc[leak.strength >= SUSPICIOUS, "feature"].tolist() or "none")

# (b) Process checks (tick these off in your report)
print()
print("[x] Split done BEFORE imputation/scaling/encoding; preprocessors fitted on train only")
print("[x] Test set saved and not used for EDA, leakage analysis or balancing")
print("[x] Identifier columns (PatientID, DoctorInCharge) removed")
print("[x] Same patient cannot appear in both sets:", ONE_ROW_PER_PATIENT)

### Leakage decision  *(write this yourself, using the table above)*
Three feature sets are defined in Stage 2 so you can show **what changes** when closer-to-diagnosis information is added:

| Set | Contents | Intended role |
|---|---|---|
| A_screening | demographics, lifestyle, medical history, clinical measurements | honest early **screening** setting |
| B_plus_symptoms | A + reported symptoms | screening with symptom questionnaire |
| C_plus_cognitive_tests | B + MMSE, FunctionalAssessment, ADL | **decision-support** setting (close to the diagnosis itself) |

Report, for each set, whether any feature looks like a leak and how you will treat it. A score such as MMSE is not a leak in the data-engineering sense (it exists before the diagnosis), but if it nearly defines the label, a model using it is reading the answer rather than assessing risk. Comparing A, B and C in Stages 9-11 turns this into a result.

### Stage 4: What / Why / How / Result  *(write this yourself)*
- **Missing values / outliers / encoding / scaling:** what you found, what you did, why.
- **Split strategy:** stratified 80/20, fixed seed, CV inside train; why (small-ish data, need a truly unseen test set).
- **Result:** features before and after preprocessing, leakage findings.

---
# Stage 5: Data Balancing

Steps: (1) measure the imbalance, (2) decide whether it needs treatment, (3) if comparing treatments, do it **only inside the training folds**.
The test set is never resampled.

In [ ]:
dist = pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "train_%": (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
    "test_count": y_test.value_counts().sort_index(),
    "test_%": (y_test.value_counts(normalize=True).sort_index() * 100).round(2),
})
dist.index = dist.index.map({0: "0 (no diagnosis)", 1: "1 (diagnosis)"})
display(dist)
ratio = y_train.value_counts().max() / y_train.value_counts().min()
print(f"Imbalance ratio (majority:minority) in train = {ratio:.2f} : 1")

fig, ax = plt.subplots(figsize=(4.5, 3.2))
sns.countplot(x=y_train, ax=ax, color="#8ab4d9")
ax.set_xticks([0, 1]); ax.set_xticklabels(["No diagnosis", "Diagnosis"]); ax.set_xlabel(""); ax.set_title("Training-set class distribution")
savefig("05_class_distribution")

**Why accuracy is not enough here:** if the majority class is the large majority, a model that predicts "no diagnosis" for everyone scores a high accuracy and catches nobody. For a screening tool the cost of a missed case is higher than the cost of a false alarm, so recall, F1, PR-AUC and balanced accuracy are tracked.

**Experiment:** compare four strategies using 5-fold stratified cross-validation on the **training set**, with resampling performed *inside* each training fold through a pipeline, so validation folds are never resampled or seen by the sampler.
- `none`: baseline, no treatment
- `class_weight`: cost-sensitive learning (no change to the data)
- `smote`: synthetic minority oversampling (needs `imbalanced-learn`)
- `undersample`: random undersampling of the majority class (needs `imbalanced-learn`)

Install if needed: `pip install imbalanced-learn`

In [ ]:
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("imbalanced-learn not installed: SMOTE/undersampling rows will be skipped. pip install imbalanced-learn")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"pr_auc": "average_precision", "roc_auc": "roc_auc", "recall": "recall",
           "precision": "precision", "f1": "f1", "balanced_acc": "balanced_accuracy"}

def make_model(name, class_weight=None):
    if name == "LogReg":
        return LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=SEED)
    return RandomForestClassifier(n_estimators=200, class_weight=class_weight, random_state=SEED, n_jobs=-1)

def make_pipeline(features, model_name, strategy):
    prep_ = build_preprocessor(features)
    if strategy == "none":
        return Pipeline([("prep", prep_), ("clf", make_model(model_name))])
    if strategy == "class_weight":
        return Pipeline([("prep", prep_), ("clf", make_model(model_name, "balanced"))])
    if strategy == "smote":
        return ImbPipeline([("prep", prep_), ("smote", SMOTE(random_state=SEED)), ("clf", make_model(model_name))])
    if strategy == "undersample":
        return ImbPipeline([("prep", prep_), ("under", RandomUnderSampler(random_state=SEED)), ("clf", make_model(model_name))])

strategies = ["none", "class_weight"] + (["smote", "undersample"] if HAS_IMBLEARN else [])
results = []
for set_name, feats in FEATURE_SETS.items():
    for model_name in ["LogReg", "RandomForest"]:
        for strat in strategies:
            pipe = make_pipeline(feats, model_name, strat)
            cvres = cross_validate(pipe, X_train[feats], y_train, cv=cv, scoring=scoring, n_jobs=1)
            row = {"feature_set": set_name, "model": model_name, "strategy": strat}
            for m in scoring:
                row[m] = f"{cvres['test_' + m].mean():.3f} +/- {cvres['test_' + m].std():.3f}"
            results.append(row)

balance_results = pd.DataFrame(results)
display(balance_results)
balance_results.to_csv(OUT_DIR / "stage5_balancing_cv_results.csv", index=False)

### Stage 5: What / Why / How / Result  *(write this yourself)*
Decide using the table, not by habit:
- Does any strategy improve **recall / F1 / PR-AUC / balanced accuracy** by more than the fold-to-fold spread (the `+/-`)? If the differences are inside the spread, the honest conclusion is that balancing makes no reliable difference.
- If the imbalance is moderate and strategies perform alike, prefer the simplest one (`class_weight`): it changes no data and has no leakage risk.
- Note the limitation of SMOTE here: most features are binary or coded categories, so interpolated synthetic rows can take fractional values that do not exist in reality (SMOTENC is the alternative if you pursue it).
- Whatever you choose, state it and carry it into Stages 8-11. Threshold tuning on the validation folds is a further option to discuss for recall.

---
# Stage 6: Exploratory Data Analysis (training set only)

Each block answers a question and ends with something you will **do** because of it.

In [ ]:
eda = X_train.copy()
eda[TARGET] = y_train.values
POS, NEG = eda[eda[TARGET] == 1], eda[eda[TARGET] == 0]
print("EDA rows (train only):", len(eda))

### 6.1 Univariate analysis: what do the distributions look like?

In [ ]:
desc = eda[CONTINUOUS].describe().T
desc["skew"] = eda[CONTINUOUS].skew()
display(desc.round(2))

ncols = 5; nrows = int(np.ceil(len(CONTINUOUS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 2.8 * nrows))
for ax, c in zip(axes.ravel(), CONTINUOUS):
    sns.histplot(eda[c], kde=True, ax=ax, color="#5b8fb9", bins=25)
    ax.set_xlabel(c); ax.set_ylabel("")
for ax in axes.ravel()[len(CONTINUOUS):]:
    ax.axis("off")
savefig("06_univariate_continuous")

prev = (eda[BINARY].mean() * 100).sort_values()
fig, ax = plt.subplots(figsize=(7, 0.32 * len(prev) + 1))
prev.plot.barh(ax=ax, color="#5b8fb9"); ax.set_xlabel("% of patients with value = 1")
ax.set_title("Prevalence of binary features (train)")
savefig("06_binary_prevalence")

### 6.2 Target distribution
See Stage 5 (class distribution plot and table).

### 6.3 Bivariate: which continuous features differ between the classes?
Mann-Whitney U (no normality assumption) with an effect size (rank-biserial correlation, range -1 to 1; the sign shows which group has higher values) and a Bonferroni-adjusted p-value. With a few thousand rows almost anything is "significant", so rely on the **effect size**, not the p-value.

In [ ]:
rows = []
for c in CONTINUOUS:
    a, b = POS[c].dropna(), NEG[c].dropna()
    U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rbc = 2 * U / (len(a) * len(b)) - 1                     # >0: diagnosed group tends higher
    pooled = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    d = (a.mean() - b.mean()) / pooled if pooled > 0 else np.nan
    rows.append({"feature": c, "median_diagnosed": a.median(), "median_not": b.median(),
                 "rank_biserial": rbc, "cohens_d": d, "p_value": p,
                 "p_bonferroni": min(p * len(CONTINUOUS), 1.0)})
cont_tbl = pd.DataFrame(rows)
cont_tbl["abs_effect"] = cont_tbl.rank_biserial.abs()
cont_tbl = cont_tbl.sort_values("abs_effect", ascending=False).drop(columns="abs_effect").reset_index(drop=True)
display(cont_tbl.round(4))

top = cont_tbl.feature.head(8).tolist()
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), top):
    sns.boxplot(data=eda, x=TARGET, y=c, ax=ax, palette=["#8ab4d9", "#e0876a"], hue=TARGET, legend=False)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["No", "Yes"]); ax.set_xlabel("Diagnosis")
savefig("06_top_continuous_by_target")

### 6.4 Bivariate: which binary / categorical features are associated with the diagnosis?

In [ ]:
def cramers_v(ct):
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    k = min(ct.shape) - 1
    return np.sqrt(chi2 / (n * k)) if k > 0 and n > 0 else np.nan

rows = []
for c in BINARY + ORDINAL + NOMINAL:
    ct = pd.crosstab(eda[c], eda[TARGET])
    chi2, p, _, _ = stats.chi2_contingency(ct, correction=False)
    rows.append({"feature": c, "levels": ct.shape[0], "cramers_v": cramers_v(ct), "p_value": p,
                 "p_bonferroni": min(p * len(BINARY + ORDINAL + NOMINAL), 1.0)})
cat_tbl = pd.DataFrame(rows).sort_values("cramers_v", ascending=False).reset_index(drop=True)
display(cat_tbl.round(4))

top_cat = cat_tbl.feature.head(8).tolist()
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, c in zip(axes.ravel(), top_cat):
    rate = eda.groupby(c)[TARGET].mean() * 100
    rate.plot.bar(ax=ax, color="#5b8fb9"); ax.set_ylabel("% diagnosed"); ax.set_xlabel(c)
    ax.axhline(eda[TARGET].mean() * 100, color="red", ls="--", lw=1)
fig.suptitle("Diagnosis rate by category (red line = overall rate)", y=1.02)
savefig("06_diagnosis_rate_by_category")

### 6.5 Age and diagnosis: is there a trend?

In [ ]:
bins = np.arange(eda.Age.min() // 5 * 5, eda.Age.max() + 5, 5)
eda["AgeBand"] = pd.cut(eda.Age, bins=bins, right=False)
age_tbl = eda.groupby("AgeBand", observed=True)[TARGET].agg(["count", "mean"])
age_tbl["mean"] = (age_tbl["mean"] * 100).round(2)
age_tbl.columns = ["n_patients", "diagnosis_rate_%"]
display(age_tbl)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(age_tbl.index.astype(str), age_tbl["diagnosis_rate_%"], marker="o", color="#e0876a")
ax.set_ylabel("% diagnosed"); ax.set_xlabel("Age band"); ax.set_title("Diagnosis rate by age band (train)")
plt.xticks(rotation=45)
savefig("06_age_trend")

### 6.6 Correlation and multicollinearity among continuous features

In [ ]:
corr = eda[CONTINUOUS].corr(method="spearman")
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax,
            annot=True, fmt=".2f", annot_kws={"size": 7}, cbar_kws={"shrink": 0.7})
ax.set_title("Spearman correlation (continuous features, train)")
savefig("06_correlation_heatmap")

pairs = (corr.where(~mask).stack().rename("rho").reset_index()
         .rename(columns={"level_0": "feature_1", "level_1": "feature_2"}))
pairs = pairs[pairs.rho.abs() >= 0.5].sort_values("rho", key=abs, ascending=False)
print("Feature pairs with |rho| >= 0.5:", "none" if pairs.empty else "")
if not pairs.empty:
    display(pairs.round(3))

# Variance inflation factors (VIF) computed from the inverse correlation matrix
Z = eda[CONTINUOUS].fillna(eda[CONTINUOUS].median())
R = np.corrcoef(Z.values.T)
vif = pd.Series(np.diag(np.linalg.pinv(R)), index=CONTINUOUS, name="VIF").sort_values(ascending=False)
display(vif.round(2).to_frame())
print("Rule of thumb: VIF above ~5-10 signals problematic multicollinearity (matters for logistic regression).")

### 6.7 Anomaly and consistency checks

In [ ]:
checks = {
    "Systolic BP <= Diastolic BP": int((eda.SystolicBP <= eda.DiastolicBP).sum()),
    "LDL + HDL > Total cholesterol": int(((eda.CholesterolLDL + eda.CholesterolHDL) > eda.CholesterolTotal).sum()),
    "Age outside 60-90 (verify expected range)": int(((eda.Age < 60) | (eda.Age > 90)).sum()),
}
display(pd.Series(checks, name="rows flagged").to_frame())
print("Flagged rows are not automatically errors: report them, check against the dataset documentation, and decide.")

### 6.8 EDA observations that change the pipeline  *(write this yourself from the outputs above)*

| # | Observation (cite the table/plot) | Decision it drives | Stage affected |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |

Prompts to guide you:
- Which features show a **meaningful effect size**, and which show essentially none? Is that plausible clinically? (If risk factors such as BMI, smoking or cholesterol show no relation at all, think about what that implies about how the data was generated.)
- Do MMSE / FunctionalAssessment / ADL separate the classes far more strongly than everything else? That is the leakage question from Stage 4 and it justifies running feature sets A, B and C.
- Any skew that suggests a transformation? Any pair with high |rho| or VIF that suggests dropping a feature or using regularisation?
- Does the diagnosis rate change with age as expected?

### Stage 6: What / Why / How / Result  *(write this yourself)*

---
# Hand-off to Stage 7 onward

In [ ]:
handoff = {
    "seed": SEED,
    "target": TARGET,
    "continuous": CONTINUOUS, "binary": BINARY, "ordinal": ORDINAL, "nominal": NOMINAL,
    "feature_sets": FEATURE_SETS,
}
with open(OUT_DIR / "feature_config.json", "w") as f:
    json.dump(handoff, f, indent=2)

print("Saved to", OUT_DIR.resolve())
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(OUT_DIR))